In [ ]:
# Copyright (c) 2026 Nokia Bell Labs
# Licensed under the BSD 3 Clause license
# SPDX-License-Identifier: BSD-3-Clause

In [ ]:
import json
from collections import defaultdict
import matplotlib.pyplot as plt
import pandas as pd
import os
import sys
import numpy as np

In [ ]:
def parse_files_in_directory(directory):
    data = defaultdict(dict)  # Dictionary to hold the parsed data for all files

    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            filepath = os.path.join(directory, filename)

            with open(filepath, "r") as file:
                file_data = {"pairs": defaultdict(list), "json_data": None}
                lines = file.readlines()

                # Process all lines except the JSON part
                json_lines = []
                for line in lines:
                    line = line.strip()
                    if line.startswith("key="):
                        # Extract key and value while accounting for "=" and "," in the content
                        try:
                            prefix, key_value_pair = line.split("key=", 1)
                            key, value = key_value_pair.split(", value=", 1)
                            # Default dict auto handles initialization to empty list
                            file_data["pairs"][key].append(value)
                        except ValueError:
                            print(f"Skipping malformed line: {line}")
                    else:
                        # Collect all JSON lines
                        json_lines.append(line)

                # Combine JSON lines and parse them
                if json_lines:
                    try:
                        combined_json = "\n".join(json_lines)
                        file_data["json_data"] = json.loads(combined_json)
                    except json.JSONDecodeError:
                        print(f"Skipping invalid JSON in file: {filename}")

                data[filename] = file_data

    return data

In [ ]:
class StatClass:
    def __init__(self, base_path, results_dir='results', exp_drv_dir='experiment_driver', drvr_file_name='experiment_list.csv'):
        self.df = None
        self.base_path = base_path
        full_input_file_path = os.path.join(self.base_path, exp_drv_dir, drvr_file_name)

        if not os.path.exists(full_input_file_path):
            print("There is no driver file so exiting out")
            sys.exit(1)
        else:
            self.df = pd.read_csv(full_input_file_path)

        self.results_dir = os.path.join(self.base_path, results_dir)

    def get_file_content(self, exps_list=[], same_ds=False):
        data_list = []
        try:
            if self.df is not None:
                for index, row in self.df.iterrows():
                    exp_id_int = row['Experiment ID']
                    if exp_id_int not in exps_list:
                        continue

                    experiment_id = str(exp_id_int)
                    result_path = os.path.join(self.results_dir, experiment_id)
                    result_data = parse_files_in_directory(result_path)
                    data_list.append({'id': experiment_id, 'results': result_data, 'meta_data': row})
            else:
                print("Data has not been loaded yet. Please 'read_file' first.")
                return None
        except ValueError:
            print("Error parsing file.")
            return None
        
        return data_list

In [ ]:
def extract_key_data_from_results(results, keys):
    """
    Extracts data for one or more keys from all files in the results structure.
    
    Parameters:
        results (list): List of result dictionaries returned by get_file_content
        keys (str or list): Single key or list of keys to extract from the pairs dictionary
        
    Returns:
        list: List of dictionaries with exp_id, filename, and values for each key
    """
    # Convert single key to list for uniform processing
    if isinstance(keys, str):
        keys = [keys]
    
    extracted_data = []
    
    for result in results:
        exp_id = result.get('id')
        results_dict = result.get('results', {})
        
        for filename, file_data in results_dict.items():
            pairs_dict = file_data.get('pairs', {})
            
            for key in keys:
                if key in pairs_dict:
                    values = pairs_dict[key]
                    extracted_data.append({
                        'exp_id': exp_id,
                        'filename': filename,
                        'key': key,
                        'values': values
                    })
    
    return extracted_data


def extract_all_keys_from_file(results):
    """
    Extracts all key-value pairs from all files in the results structure.
    
    Parameters:
        results (list): List of result dictionaries returned by get_file_content
        
    Returns:
        list: List of dictionaries containing all pairs data for each file in each experiment
    """
    extracted_data = []
    
    for result in results:
        exp_id = result.get('id')
        results_dict = result.get('results', {})
        
        for filename, file_data in results_dict.items():
            pairs_dict = file_data.get('pairs', {})
            
            extracted_data.append({
                'exp_id': exp_id,
                'filename': filename,
                'all_pairs': dict(pairs_dict)
            })
    
    return extracted_data


def get_json_data_from_file(results):
    """
    Extracts JSON data from all files in the results structure.
    
    Parameters:
        results (list): List of result dictionaries returned by get_file_content
        
    Returns:
        list: List of dictionaries containing JSON data for each file in each experiment
    """
    extracted_data = []
    
    for result in results:
        exp_id = result.get('id')
        results_dict = result.get('results', {})
        
        for filename, file_data in results_dict.items():
            json_data = file_data.get('json_data')
            
            if json_data is not None:
                extracted_data.append({
                    'exp_id': exp_id,
                    'filename': filename,
                    'json_data': json_data
                })
    
    return extracted_data

In [ ]:
def plot_extracted_data(extracted_data):
    """
    Plots the extracted data with step numbers on X-axis and values on Y-axis.
    Groups data by file and plots multiple keys as different lines on the same subplot.
    
    Parameters:
        extracted_data (list): List of dictionaries from extract_key_data_from_results
    """
    if not extracted_data:
        print("No data to plot")
        return
    
    # Group data by exp_id and filename
    grouped_data = {}
    for data_item in extracted_data:
        exp_id = data_item['exp_id']
        filename = data_item['filename']
        key = (exp_id, filename)
        
        if key not in grouped_data:
            grouped_data[key] = []
        grouped_data[key].append(data_item)
    
    num_plots = len(grouped_data)
    fig, axes = plt.subplots(1, num_plots, figsize=(8*num_plots, 6))
    
    # Handle single plot case
    if num_plots == 1:
        axes = [axes]
    
    for idx, ((exp_id, filename), items) in enumerate(grouped_data.items()):
        ax = axes[idx]
        
        # Plot each key as a separate line
        for data_item in items:
            key = data_item['key']
            values_raw = data_item['values']
            
            # Parse the values to extract step and value
            steps = []
            values = []
            for val_str in values_raw:
                try:
                    # Split by ', step=' to get value and step
                    parts = val_str.split(', step=')
                    if len(parts) == 2:
                        value = float(parts[0])
                        step = int(parts[1])
                        steps.append(step)
                        values.append(value)
                except (ValueError, IndexError) as e:
                    print(f"Skipping malformed value: {val_str}, error: {e}")
            
            # Plot the line for this key
            ax.plot(steps, values, marker='o', linewidth=2, markersize=6, label=key)
        
        ax.set_xlabel('Rounds', fontsize=12)
        ax.set_ylabel('Adjusted Rand Index', fontsize=12)
        #ax.set_title(f'Exp: {exp_id}\n{filename}', fontsize=11, fontweight='bold')
        #ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
def extracted_data_to_dataframe(extracted_data):
    """
    Converts extracted data to a pandas DataFrame with step and value columns.
    
    Parameters:
        extracted_data (list): List of dictionaries from extract_key_data_from_results
        
    Returns:
        pd.DataFrame: DataFrame with columns: exp_id, filename, key, step, value
    """
    rows = []
    
    for data_item in extracted_data:
        exp_id = data_item['exp_id']
        filename = data_item['filename']
        key = data_item['key']
        values_raw = data_item['values']
        
        # Parse the values to extract step and value
        for val_str in values_raw:
            try:
                # Split by ', step=' to get value and step
                parts = val_str.split(', step=')
                if len(parts) == 2:
                    value = float(parts[0])
                    step = int(parts[1])
                    rows.append({
                        'exp_id': exp_id,
                        'filename': filename,
                        'key': key,
                        'step': step,
                        'value': value
                    })
            except (ValueError, IndexError) as e:
                print(f"Skipping malformed value: {val_str}, error: {e}")
    
    df = pd.DataFrame(rows)
    return df


def extracted_data_to_wide_dataframe(extracted_data):
    """
    Converts extracted data to a wide pandas DataFrame where each row is a file+key combination
    and columns are steps with their corresponding values arranged horizontally.
    
    Parameters:
        extracted_data (list): List of dictionaries from extract_key_data_from_results
        
    Returns:
        pd.DataFrame: DataFrame with exp_id, filename, key as identifiers and 
                     step_0, step_1, step_2, ... as columns containing values.
                     One row per file+key combination.
    """
    rows = []
    
    for data_item in extracted_data:
        exp_id = data_item['exp_id']
        filename = data_item['filename']
        key = data_item['key']
        values_raw = data_item['values']
        
        # Create a row dictionary with metadata
        row_dict = {
            'exp_id': exp_id,
            'filename': filename,
            'key': key
        }
        
        # Parse the values and add them as step_N columns
        for val_str in values_raw:
            try:
                # Split by ', step=' to get value and step
                parts = val_str.split(', step=')
                if len(parts) == 2:
                    value = float(parts[0])
                    step = int(parts[1])
                    row_dict[f'step_{step}'] = value
            except (ValueError, IndexError) as e:
                print(f"Skipping malformed value: {val_str}, error: {e}")
        
        rows.append(row_dict)
    
    df = pd.DataFrame(rows)
    
    # Sort columns: first the metadata columns, then step columns in order
    metadata_cols = ['exp_id', 'filename', 'key']
    step_cols = sorted([col for col in df.columns if col.startswith('step_')], 
                       key=lambda x: int(x.split('_')[1]))
    df = df[metadata_cols + step_cols]
    
    return df



In [ ]:
def save_dataframe_to_csv(df, filename, output_dir='../results'):
    """
    Saves a pandas DataFrame to a CSV file with numeric values rounded to 3 decimal places.
    
    Parameters:
        df (pd.DataFrame): The DataFrame to save
        filename (str): Name of the output CSV file (e.g., 'results.csv')
        output_dir (str): Directory path where the file will be saved (default: '../results')
        
    Returns:
        str: Full path of the saved file
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Create full file path
    filepath = os.path.join(output_dir, filename)
    
    # Round numeric columns to 3 decimal places
    df_rounded = df.copy()
    numeric_cols = df_rounded.select_dtypes(include=[np.number]).columns
    df_rounded[numeric_cols] = df_rounded[numeric_cols].round(3)
    
    # Save DataFrame to CSV
    df_rounded.to_csv(filepath, index=False)
    
    print(f"DataFrame saved to: {filepath}")
    return filepath



In [ ]:
directory_path = "../"
results_dir='results'
statter = StatClass(directory_path, results_dir)
base_log_dir = os.path.join(directory_path, results_dir)

In [ ]:
# With early stopping
exp = 4003
exps_to_include = [exp]
results = statter.get_file_content(exps_list=exps_to_include)
# Can pass a single key or a list of keys
keys = [ 'adjusted_rand_score_test']  # or just 'adjusted_rand_score_train' for single key
#keys = ['rand_score_train', 'rand_score_test']
out_file = f"rand_score_{exp}.csv"
extracted_data = extract_key_data_from_results(results, keys=keys)
# Create wide DataFrame from extracted data
df_extracted_wide = extracted_data_to_wide_dataframe(extracted_data)
# Example: Save the wide DataFrame to CSV
output_path = save_dataframe_to_csv(df_extracted_wide, out_file)
# Plot the extracted data
fig = plot_extracted_data(extracted_data)
fig.savefig("pp_pathological_1.pdf", format='pdf')

In [ ]:
# Without early stopping
exp = 4004
exps_to_include = [exp]
results = statter.get_file_content(exps_list=exps_to_include)
# Can pass a single key or a list of keys
keys = [ 'adjusted_rand_score_test']  # or just 'adjusted_rand_score_train' for single key
#keys = ['rand_score_train', 'rand_score_test']
out_file = f"rand_score_{exp}.csv"
extracted_data = extract_key_data_from_results(results, keys=keys)
# Create wide DataFrame from extracted data
df_extracted_wide = extracted_data_to_wide_dataframe(extracted_data)
# Example: Save the wide DataFrame to CSV
output_path = save_dataframe_to_csv(df_extracted_wide, out_file)
# Plot the extracted data
fig = plot_extracted_data(extracted_data)
fig.savefig("pp_pathological_2.pdf", format='pdf')

In [ ]:
# Debug: Check what keys were extracted
print(f"Number of items in extracted_data: {len(extracted_data)}")
for item in extracted_data:
    print(f"Exp: {item['exp_id']}, File: {item['filename']}, Key: {item['key']}, Num values: {len(item['values'])}")

In [ ]:
# Debug: Check the DataFrame structure
print(f"DataFrame shape: {df_extracted_wide.shape}")
print(f"\nDataFrame columns: {df_extracted_wide.columns.tolist()}")
print(f"\nUnique keys in DataFrame: {df_extracted_wide['key'].unique()}")
print(f"\nDataFrame head:")
print(df_extracted_wide.head(10))